In [1]:
import pandas as pd

# Loads every sheet into a dict: {"Sheet1": df1, "Sheet2": df2, ...}
all_sheets = pd.read_excel("DV.xlsx", sheet_name=None)
for name, df in all_sheets.items():
    df.to_csv(f"{name}.csv", index=False)

In [2]:
import numpy as np
import re

df_eg = pd.read_csv("Energy, GHG.csv", header=None)

print(f"Kích thước dữ liệu gốc: {df_eg.shape}")
display(df_eg.head())

Kích thước dữ liệu gốc: (133, 22)


,0,1,2,3,4,5,6,7,8,9,...,12,13,14,15,16,17,18,19,20,21
0,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21
1,Energy Consumption,unit,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,...,Nov,Dec,YTD,Q1,Q2,Q3,Q4,NaN,NaN,NaN
2,Coal/ Than 5A (DV),Ton,1233.5220000000002,983,2414.661,2198.795,2197.301,2170.99,2442.916,2148.882,...,1649.3,2061.215,NaN,4631.183,6567.086,6310.711,5464.47,NaN,NaN,NaN
3,Coal/ Than 4A.1 (DV),Ton,443,251.25,543.069,88.335,774.19,319.418,508.88,630.609,...,113.76,369.092,NaN,1237.319,1181.943,1584.048,501.752,NaN,NaN,NaN
4,Coal/ Than import coal (DV) (than nhập khẩu),Ton,0,92.48,430.117,722.734,0,434.63,375.27,77.21,...,352.255,533.023,NaN,522.597,1157.364,452.48,1436.158,NaN,NaN,NaN


In [3]:
# Tìm index của các dòng chứa chữ 'unit'
header_rows = df_eg[df_eg[1].astype(str).str.strip().str.lower() == 'unit'].index.tolist()
print("Vị trí các dòng Header được tìm thấy:", header_rows)

# Cắt DataFrame lớn thành dictionary chứa các DataFrame nhỏ
tables = {}
for i in range(len(header_rows)):
    start_idx = header_rows[i]

    # Lấy tên bảng từ cột đầu tiên
    table_name = str(df_eg.iloc[start_idx, 0]).strip()

    # Xác định điểm kết thúc của bảng này
    end_idx = header_rows[i+1] if i + 1 < len(header_rows) else len(df_eg)

    # Cắt lấy dữ liệu và drop các dòng trống hoàn toàn
    df_sub = df_eg.iloc[start_idx:end_idx].dropna(how='all').reset_index(drop=True)
    tables[table_name] = df_sub

print(f"Đã tách thành công {len(tables)} bảng: {list(tables.keys())}")

Vị trí các dòng Header được tìm thấy: [1, 67, 119]
Đã tách thành công 3 bảng: ['Energy Consumption', 'Greenhouse Gas Emission', 'Energy Cost']


In [4]:
viet_chars = (
    r'àáâãèéêìíòóôõùúýăđơư'
    r'ạảấầẩẫậắằẳẵặẹẻẽếềểễệ'
    r'ỉịọỏốồổỗộớờởỡợụủứừửữự'
    r'ỳỵỷỹÀÁÂÃÈÉÊÌÍÒÓÔÕÙÚÝĂĐƠƯ'
)

def remove_vietnamese(text):
    if not isinstance(text, str) or pd.isna(text):
        return text
    # 1. Xóa phần trong ngoặc chứa tiếng Việt
    text = re.sub(rf'\s*\([^)]*[{viet_chars}][^)]*\)', '', text)
    # 2. Xóa phần sau dấu gạch chéo '/' chứa tiếng Việt
    text = re.sub(rf'\s*/[^/]*[{viet_chars}].*', '', text)
    # 3. Xóa các cụm từ tiếng Việt đứng trơ trọi ở cuối chuỗi (Vd: "Dầu xe")
    text = re.sub(rf'\s+[{viet_chars}a-zA-Z]*[{viet_chars}][\w\s]*$', '', text)
    return text.strip()

In [5]:
cleaned_tables = []
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

for table_name, df_sub in tables.items():
    # Lấy dòng đầu làm Header
    df_sub.columns = [str(c).strip() for c in df_sub.iloc[0]]
    df_sub = df_sub[1:].reset_index(drop=True)

    # Cột nào có tên thì lấy (Lọc bỏ cột Unnamed hoặc nan)
    valid_cols = [c for c in df_sub.columns if not c.lower().startswith('unnamed') and c.lower() != 'nan']
    df_sub = df_sub[valid_cols]

    # Cố định tên 2 cột đầu
    cols = list(df_sub.columns)
    if len(cols) >= 2:
        cols[0] = "original_name"
        cols[1] = "unit"
        df_sub.columns = cols

    # Không có tên thì bỏ hàng đó đi (Drop NaN/chuỗi rỗng ở cột đầu tiên)
    df_sub = df_sub.dropna(subset=['original_name'])
    df_sub = df_sub[df_sub['original_name'].astype(str).str.strip() != '']
    df_sub = df_sub[df_sub['original_name'].astype(str).str.strip().str.lower() != 'nan']

    # Xác định các cột tháng
    valid_months = [m for m in months if m in df_sub.columns]

    if valid_months:
        # Chuyển Wide to Long
        df_long = df_sub.melt(
            id_vars=["original_name", "unit"],
            value_vars=valid_months,
            var_name="period",
            value_name="value"
        )

        df_long.insert(0, "category", table_name)

        # Xóa tiếng Việt
        df_long.insert(1, "name", df_long["original_name"].apply(remove_vietnamese))
        cleanup_rules = {
            r'Than cám qua sàng / Than ': 'Coal ',  # Đổi Than cám thành Coal (áp dụng cho cả YB, DV, DV+YB)
            r'/ Than non': '',                      # Cắt bỏ đuôi / Than non
            r'/ Than ': ' ',                        # Cắt bỏ / Than
        }
        for old_text, new_text in cleanup_rules.items():
            df_long["name"] = df_long["name"].str.replace(old_text, new_text, regex=True)

        df_long["name"] = df_long["name"].str.strip()
        df_long["unit"] = df_long["unit"].astype(str).str.strip()

        # Lấp khoảng trống bằng 0
        df_long["value"] = df_long["value"].astype(str).str.strip()
        df_long["value"] = df_long["value"].replace(['-', '', 'nan'], '0') # Trống biến thành 0
        df_long["value"] = df_long["value"].str.replace(',', '', regex=False)
        df_long["value"] = pd.to_numeric(df_long["value"], errors="coerce").fillna(0).round(3)

        cleaned_tables.append(df_long)
print(f"Đã transform xong {len(cleaned_tables)} bảng con. Mọi tên nguyên liệu đã được chuẩn hóa!")

Đã transform xong 3 bảng con. Mọi tên nguyên liệu đã được chuẩn hóa!


In [8]:
# Nối tất cả các bảng đã làm sạch thành 1 bảng Master duy nhất
final_df = pd.concat(cleaned_tables, ignore_index=True)

# Sắp xếp lại thứ tự các cột cho logic
final_df = final_df[['category', 'name', 'original_name', 'unit', 'period', 'value']]

# Xem tổng quan dữ liệu
print("--- THÔNG TIN DỮ LIỆU ---")
final_df.info()

print("\n--- XEM TRƯỚC DỮ LIỆU ---")
display(final_df.head(50))

# Xuất ra file CSV
final_df.to_csv("Energy_GHG_cleaned.csv", index=False)
print("\n Dữ liệu sạch đã được lưu ra file ")

--- THÔNG TIN DỮ LIỆU ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   category       1500 non-null   object 
 1   name           1500 non-null   object 
 2   original_name  1500 non-null   object 
 3   unit           1500 non-null   object 
 4   period         1500 non-null   object 
 5   value          1500 non-null   float64
dtypes: float64(1), object(5)
memory usage: 70.4+ KB

--- XEM TRƯỚC DỮ LIỆU ---


,category,name,original_name,unit,period,value
0,Energy Consumption,Coal 5A (DV),Coal/ Than 5A (DV),Ton,Jan,1233.522
1,Energy Consumption,Coal 4A.1 (DV),Coal/ Than 4A.1 (DV),Ton,Jan,443.000
2,Energy Consumption,Coal import coal (DV),Coal/ Than import coal (DV) (than nhập khẩu),Ton,Jan,0.000
3,Energy Consumption,Coal 5A under screen DV,Than cám qua sàng / Than 5A under screen DV,Ton,Jan,114.000
4,Energy Consumption,Coal 5A (YB),Coal/ Than 5A (YB),Ton,Jan,0.000
5,Energy Consumption,Coal 4A.1 (YB),Coal/ Than 4A.1 (YB),Ton,Jan,0.000
6,Energy Consumption,Coal import coal (YB),Coal/ Than import coal (YB) (than nhập khẩu),Ton,Jan,0.000
7,Energy Consumption,Coal 5A under screen YB,Than cám qua sàng / Than 5A under screen YB,Ton,Jan,0.000
8,Energy Consumption,Coal 5A (DV+YB),Coal/ Than 5A (DV+YB),Ton,Jan,1233.522
9,Energy Consumption,Coal 4A.1 (DV+YB),Coal/ Than 4A.1 (DV+YB),Ton,Jan,443.000



 Dữ liệu sạch đã được lưu ra file 
